# FF3 Security Selection — 단일 구간 2000-01 ~ 2010-12

- 구간 내내 상장된 종목(balanced panel), 종목별 FF3 회귀 1회, **t(α) 내림차순**
- 게이트: α>0 & t(α)≥2 & 유동성 하위 20% 제외 & sigma guard
- **2011년 이후 정보 차단**: 로드 직후 2010-12 이후 삭제, 종목명·티커는 2010-12 기준, 구간 밖 검증 없음

Colab 에서는 첫 셀의 `BASE` 를 Drive 경로로 바꾸세요.

In [ ]:
# =========================================================================
# Fama-French 3-Factor 기반 Security Selection — 단일 구간 2000-01 ~ 2010-12
# =========================================================================
# 설계
#   구간      : 2000-01 ~ 2010-12 (주가 132개월, 월수익률 131개)
#   유니버스  : 구간 내내 상장을 유지한 종목 (balanced panel)
#               - 구간의 모든 월에 ADJ_PRC 관측치가 있고 모든 월수익률이 유효
#               - 2010-12 시점 EXCHCD ∈ {1,2,3}
#   회귀      : R_i - RF = α_i + b_i(Mkt-RF) + s_i·SMB + h_i·HML + e_i  (구간 전체 1회)
#   게이트    : α > 0 AND t(α) ≥ T_CUT AND 유동성 하위 20% 제외 AND sigma guard 통과
#   정렬      : t(α) 내림차순. 상한 없음 (게이트 통과 전체가 결과물)
#
# 2011년 이후 정보 차단
#   (1) 주가·팩터 데이터를 로드 직후 SAMPLE_END(2010-12) 이후 행 삭제 → assert 확인
#   (2) 종목명·티커·거래소는 2010-12 시점 값 사용 (이전 코드는 2020년 마지막 값이었음)
#   (3) 구간 밖 성과 검증(OOS) 없음 — 2011년 이후 수익률을 전혀 읽지 않음
#   (4) 복수 클래스 중복제거·유동성 필터도 구간 내 거래량만 사용
#   → 실행 끝에 "사용한 데이터의 최대월" 을 출력·검증
#
# 주의 (설계에서 오는 한계, 2011년 정보와는 무관)
#   구간 내내 상장된 종목만 대상이므로 구간 중 상장폐지·피인수 종목이 빠져
#   수익률·α·Sharpe 가 상방 편의된다 (survivorship bias). 리포트에 명시할 것.
#
# 산출물 (outputs/single_2000_2010/)
#   ff3_selected.csv              게이트 통과 전체, t(α) 순
#   ff3_regression_all.csv        balanced 유니버스 전체 회귀 결과, t(α) 순
#   ff3_dedup_log.csv             복수 클래스 중복제거 내역
#   ff3_insample_performance.csv  구간 내 성과 (선정과 같은 구간 — 예측력 증거 아님)
# =========================================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

BASE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
PATH_STOCK = os.path.join(BASE, "data", "IFE_term_stock_data.csv")
PATH_FF = os.path.join(BASE, "data", "F-F_Research_Data_Factors.csv")
OUTPUT_DIR = os.path.join(BASE, "outputs", "single_2000_2010")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_START = pd.Period("2000-01", freq="M")
SAMPLE_END = pd.Period("2010-12", freq="M")

RANK_METRIC = "t_alpha"
T_CUT = 2.0
LIQ_CUT = 0.20
VALID_EXCHCD = [1, 2, 3]
RET_CAP = 5.0

DEDUP_BY_COMPANY = True     # 같은 회사 복수 클래스(예: HEICO)는 구간 평균 거래량 큰 쪽만
# 트래킹 주식은 회사명이 같아도 서로 다른 사업부를 추적하므로 합치지 않는다
TRACKING_STOCK_NAMES = {"APPLERA CORP", "LOEWS CORP"}

# sigma guard (Sharpe 폭주 종목만 결측 처리)
SIGMA_FLOOR_ABS = 0.015
ZERO_RET_MAX = 0.30
SHARPE_CAP = 3.0
SIGMA_TAIL_PCTILE = 0.05

# [가정] ADJ_PRC 는 배당 재투자 반영 조정가격(총수익률 기준)


def section(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

In [ ]:
# =========================================================================
# 1. 데이터 로드 — 2010-12 이후 즉시 삭제
# =========================================================================
section("[1] 주가 데이터 로드")
raw = pd.read_csv(PATH_STOCK, parse_dates=["date"])
raw["ym"] = raw["date"].dt.to_period("M")
n_all = len(raw)
raw = raw[(raw["ym"] >= SAMPLE_START) & (raw["ym"] <= SAMPLE_END)].copy()
assert raw["ym"].max() <= SAMPLE_END
print(f"원본 {n_all:,}행 → {SAMPLE_START}~{SAMPLE_END} {len(raw):,}행 "
      f"(2011-01 이후 {n_all - len(raw):,}행 삭제)")

dup = raw.duplicated(subset=["PERMNO", "ym"]).sum()
print(f"[체크] PERMNO-월 중복: {dup:,}")
if dup:
    raw = raw.drop_duplicates(subset=["PERMNO", "ym"], keep="first")
raw = raw[raw["ADJ_PRC"] > 0].sort_values(["PERMNO", "ym"]).reset_index(drop=True)

section("[2] Fama-French 3 팩터 로드")


def load_ff3(path):
    rows = []
    for ln in open(path, encoding="latin-1").read().splitlines():
        p = [x.strip() for x in ln.split(",")]
        if len(p) == 5 and p[0].isdigit() and len(p[0]) == 6:
            rows.append(p)
    ff = pd.DataFrame(rows, columns=["ym", "MKT_RF", "SMB", "HML", "RF"])
    ff["ym"] = pd.PeriodIndex(ff["ym"].str[:4] + "-" + ff["ym"].str[4:], freq="M")
    return ff.set_index("ym").astype(float) / 100.0


ff3 = load_ff3(PATH_FF).loc[SAMPLE_START:SAMPLE_END]
assert ff3.index.max() <= SAMPLE_END
print(f"팩터 {ff3.index.min()}~{ff3.index.max()} ({len(ff3)}개월)")

In [ ]:
# =========================================================================
# 3. Balanced panel
# =========================================================================
section("[3] Balanced panel 구성")
n_months = (SAMPLE_END - SAMPLE_START).n + 1          # 132
n_rets = n_months - 1                                 # 131

n_appear = raw["PERMNO"].nunique()
price_cnt = raw.groupby("PERMNO")["ym"].nunique()
full_price = price_cnt[price_cnt == n_months].index
sub = raw[raw["PERMNO"].isin(full_price)].copy()

g = sub.groupby("PERMNO")
gap = (sub["ym"] - g["ym"].shift(1)).apply(lambda x: x.n if pd.notna(x) else np.nan)
sub["RET"] = np.where(gap == 1, sub["ADJ_PRC"] / g["ADJ_PRC"].shift(1) - 1.0, np.nan)
n_cap = int((sub["RET"] > RET_CAP).sum())
sub.loc[sub["RET"] > RET_CAP, "RET"] = np.nan

ret_cnt = sub.dropna(subset=["RET"]).groupby("PERMNO").size()
balanced = ret_cnt[ret_cnt == n_rets].index

# 2010-12 시점 거래소 조건
at_end = sub[sub["ym"] == SAMPLE_END].set_index("PERMNO")
exch_ok = at_end.index[at_end["EXCHCD"].isin(VALID_EXCHCD)]
balanced = balanced.intersection(exch_ok)

panel = sub[sub["PERMNO"].isin(balanced)].dropna(subset=["RET"]).copy()
print(f"구간 내 등장 {n_appear:,} → 132개월 주가 보유 {len(full_price):,} → "
      f"131개월 수익률 유효·2010-12 거래소 조건 {len(balanced):,}")
print(f"[체크] RET > {RET_CAP:.0%} 제거: {n_cap}건")

# 종목 식별 정보 — 2010-12 시점 값 (2011년 이후 정보 아님)
ident = at_end.loc[balanced, ["TICKER", "COMNAM", "EXCHCD"]]
avg_vol = panel.groupby("PERMNO")["VOL"].mean()

# 복수 클래스 중복제거 (구간 평균 거래량 기준, 성과와 무관한 사전 규칙)
dedup_log = pd.DataFrame()
if DEDUP_BY_COMPANY:
    tmp = ident[["COMNAM", "TICKER"]].assign(avg_vol=avg_vol.reindex(ident.index))
    is_track = tmp["COMNAM"].isin(TRACKING_STOCK_NAMES)
    dups = tmp[~is_track & tmp["COMNAM"].duplicated(keep=False)].sort_values(
        ["COMNAM", "avg_vol"], ascending=[True, False])
    keep = dups.drop_duplicates("COMNAM", keep="first").index
    drop = dups.index.difference(keep)
    dedup_log = dups.assign(kept=dups.index.isin(keep)).reset_index()
    panel = panel[~panel["PERMNO"].isin(drop)]
    ident = ident.drop(drop)
    print(f"[중복제거] 같은 회사 복수 클래스 {dups['COMNAM'].nunique()}개사 → "
          f"{len(drop)}종목 제외 (거래량 큰 클래스 유지). "
          f"트래킹 주식 {sorted(TRACKING_STOCK_NAMES)} 은 합치지 않음")
print(f"최종 유니버스: {panel['PERMNO'].nunique():,}종목")
print("※ 구간 내내 상장된 종목만 대상 → survivorship bias 내재 (리포트에 명시)")

In [ ]:
# =========================================================================
# 4. 종목별 FF3 회귀 (전 종목 같은 X → 일괄 OLS)
# =========================================================================
section("[4] 종목별 FF3 회귀")
R = panel.pivot(index="ym", columns="PERMNO", values="RET")
months = R.index
assert R.notna().all().all() and len(months) == n_rets
F = ff3.loc[months, ["MKT_RF", "SMB", "HML"]].to_numpy()
rf = ff3.loc[months, "RF"].to_numpy()
Y = R.to_numpy() - rf[:, None]

T = len(months)
X = np.column_stack([np.ones(T), F])
XtX_inv = np.linalg.inv(X.T @ X)
B = XtX_inv @ X.T @ Y
E = Y - X @ B
s2 = (E ** 2).sum(axis=0) / (T - 4)
se = np.sqrt(np.outer(np.diag(XtX_inv), s2))
Tst = B / se
r2 = 1 - (E ** 2).sum(axis=0) / ((Y - Y.mean(axis=0)) ** 2).sum(axis=0)

res = pd.DataFrame({
    "PERMNO": R.columns,
    "alpha": B[0], "b_mkt": B[1], "s_smb": B[2], "h_hml": B[3],
    "t_alpha": Tst[0], "t_mkt": Tst[1], "t_smb": Tst[2], "t_hml": Tst[3],
    "r2": r2, "resid_sd": np.sqrt(s2), "se_alpha": se[0], "nobs": T,
    "ex_mean_m": Y.mean(axis=0), "ex_sd_m": Y.std(axis=0, ddof=1),
    "zero_ret_share": (np.abs(R.to_numpy()) < 1e-9).mean(axis=0),
})
res = res.join(ident, on="PERMNO")
res["avg_vol"] = res["PERMNO"].map(avg_vol)
res["alpha_ann"] = res["alpha"] * 12
res["p_alpha_one"] = stats.t.sf(res["t_alpha"], df=T - 4)
res["appraisal_ratio"] = res["alpha"] / res["resid_sd"]
res["sharpe_raw"] = res["ex_mean_m"] / res["ex_sd_m"] * np.sqrt(12)
print(f"회귀 {len(res):,}종목 × {T}개월 ({months[0]}~{months[-1]})")

# sigma guard
sd_tail = res["ex_sd_m"].quantile(SIGMA_TAIL_PCTILE)
flag = ((res["ex_sd_m"] < SIGMA_FLOOR_ABS) | (res["zero_ret_share"] > ZERO_RET_MAX)
        | ~np.isfinite(res["sharpe_raw"])
        | ((res["sharpe_raw"].abs() > SHARPE_CAP) & (res["ex_sd_m"] < sd_tail)))
res["sharpe_valid"] = ~flag
res["sharpe"] = res["sharpe_raw"].where(res["sharpe_valid"])
print(f"[Sigma guard] 결측 처리 {int(flag.sum())}종목")

# 유동성 & 게이트
res["liq_pass"] = res["avg_vol"] >= res["avg_vol"].quantile(LIQ_CUT)
res["gate_pass"] = ((res["alpha"] > 0) & (res["t_alpha"] >= T_CUT)
                    & res["liq_pass"] & res["sharpe"].notna())

In [ ]:
# =========================================================================
# 5. 선정 결과 (t(α) 순)
# =========================================================================
section("[5] 선정 결과 — t(α) 내림차순")
m = len(res)
p = np.sort(res["p_alpha_one"].to_numpy())
bh = np.where(p <= np.arange(1, m + 1) / m * 0.05)[0]
n_fdr = int(bh.max() + 1) if len(bh) else 0
exp_chance = m * stats.t.sf(T_CUT, df=T - 4)
gate = (res["alpha"] > 0) & (res["t_alpha"] >= T_CUT)
print(f"유니버스 {m:,} → α>0 & t(α)≥{T_CUT} {int(gate.sum())} "
      f"(우연 기대 {exp_chance:.0f}) → +유동성 {int((gate & res['liq_pass']).sum())} "
      f"→ +sigma guard {int(res['gate_pass'].sum())}")
print(f"※ 다중검정: BH-FDR 5% 생존 {n_fdr}개 | Bonferroni 임계 p = {0.05/m:.2e}")

res = res.sort_values(RANK_METRIC, ascending=False).reset_index(drop=True)
res.insert(0, "rank_all", np.arange(1, m + 1))
sel = res[res["gate_pass"]].copy()
sel.insert(0, "rank", np.arange(1, len(sel) + 1))

show = ["rank", "TICKER", "COMNAM", "t_alpha", "alpha_ann", "sharpe",
        "b_mkt", "s_smb", "h_hml", "r2"]
pd.set_option("display.width", 200)
print(f"\n게이트 통과 {len(sel)}종목 — 상위 30")
print(sel[show].head(30).round(3).to_string(index=False))

In [ ]:
# =========================================================================
# 6. 구간 내 성과 (참고용, 선정과 같은 구간)
# =========================================================================
section("[6] 구간 내 성과 (in-sample — 예측력 증거 아님)")


def perf(ex, name, n):
    Xp = np.column_stack([np.ones(len(ex)), ff3.loc[ex.index, ["MKT_RF", "SMB", "HML"]]])
    ip = np.linalg.inv(Xp.T @ Xp)
    b = ip @ Xp.T @ ex.to_numpy()
    e = ex.to_numpy() - Xp @ b
    s = np.sqrt(np.diag(ip) * (e @ e) / (len(ex) - 4))
    return {"portfolio": name, "n_names": n, "months": len(ex),
            "ex_ret_ann": ex.mean() * 12, "vol_ann": ex.std() * np.sqrt(12),
            "sharpe": ex.mean() / ex.std() * np.sqrt(12),
            "ff3_alpha_ann": b[0] * 12, "t_alpha": b[0] / s[0], "beta": b[1]}


EX = R.sub(rf, axis=0)
perf_df = pd.DataFrame([
    perf(EX[sel["PERMNO"]].mean(axis=1), "게이트 통과 전체 (EW)", len(sel)),
    perf(EX.mean(axis=1), "balanced 유니버스 (EW)", EX.shape[1]),
])
mk = ff3.loc[months, "MKT_RF"]
perf_df.loc[len(perf_df)] = {"portfolio": "Market (Mkt-RF)", "months": len(mk),
                             "ex_ret_ann": mk.mean() * 12, "vol_ann": mk.std() * np.sqrt(12),
                             "sharpe": mk.mean() / mk.std() * np.sqrt(12), "beta": 1.0}
print(perf_df.round(3).to_string(index=False))

In [ ]:
# =========================================================================
# 7. 2011년 이후 정보 차단 확인
# =========================================================================
section("[7] 2011년 이후 정보 차단 확인")
checks = {
    "로드된 주가 최대월 ≤ 2010-12": raw["ym"].max() <= SAMPLE_END,
    "팩터 최대월 ≤ 2010-12": ff3.index.max() <= SAMPLE_END,
    "회귀에 쓴 최대월 = 2010-12": months.max() == SAMPLE_END,
    "선정 종목 티커 = 원자료 2010-12 행의 티커":
        bool((sel.set_index("PERMNO")["TICKER"]
              == raw[raw["ym"] == SAMPLE_END].set_index("PERMNO")["TICKER"]
              .reindex(sel["PERMNO"])).all()),
}
for k, v in checks.items():
    print(f"  [{'OK' if v else 'FAIL'}] {k}")
assert all(checks.values())
# 참고: 이전 코드처럼 전체 데이터 마지막(2020년) 티커를 붙였다면 달라졌을 종목 수
full_last = pd.read_csv(PATH_STOCK, usecols=["PERMNO", "date", "TICKER"],
                        parse_dates=["date"]).sort_values("date").groupby("PERMNO")["TICKER"].last()
diff = int((sel.set_index("PERMNO")["TICKER"] != full_last.reindex(sel["PERMNO"])).sum())
print(f"  참고: 선정 종목 중 2010-12 티커 ≠ 2020년 마지막 티커: {diff}종목 "
      f"(이전 방식이었다면 2011년 이후 정보가 섞였을 부분)")

In [ ]:
# =========================================================================
# 8. 저장
# =========================================================================
sel.to_csv(os.path.join(OUTPUT_DIR, "ff3_selected.csv"), index=False)
res.to_csv(os.path.join(OUTPUT_DIR, "ff3_regression_all.csv"), index=False)
perf_df.to_csv(os.path.join(OUTPUT_DIR, "ff3_insample_performance.csv"), index=False)
if len(dedup_log):
    dedup_log.to_csv(os.path.join(OUTPUT_DIR, "ff3_dedup_log.csv"), index=False)
print(f"\n✔️ 산출물 저장: {OUTPUT_DIR}")